# Pré-processamento

**OBS.** Assumimos que a análise exploratória de dados (EDA) já foi feita.

Esse exemplo vai cobrir
+ Divisão do dataset
+ Escalonamento de atributos
+ Codificação de atributos
+ Engenharia de atributos
+ Extração de atributos
+ Seleção de atributos

In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder, FunctionTransformer
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.feature_selection import VarianceThreshold
from sklearn.decomposition import PCA
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score

## 1. Dataset de exemplo

Informações de pessoas que compraram ou não um determinado produto.

É um problema de classificação binária com 3 atributos.

In [ ]:
df = pd.DataFrame({
    "idade": [25, 32, 47, 51, 62],
    "salario": [3000, 4500, 8000, 9000, 12000],
    "estado": ["SP", "RJ", "SP", "MG", "RJ"],
    "comprou": [0, 1, 1, 1, 0]
})

df

,idade,salario,estado,comprou
0,25,3000,SP,0
1,32,4500,RJ,1
2,47,8000,SP,1
3,51,9000,MG,1
4,62,12000,RJ,0


## 2. Separação de variáveis explicativas, i.e., atributos, (X) e alvo (y), i.e., rótulo.

In [ ]:
X = df.drop("comprou", axis=1)
y = df["comprou"]

**OBS**.: O rótulo identifica dois grupos, ou classes, classe de quem comprou, `1` e a classe de quem não comprou, `0`.

## 3. Divisão do dataset

`test_size` define o tamanho do subconjunto de teste.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

### Dimensões dos conjuntos de treinamento e teste.

In [ ]:
X_train.shape

(4, 3)

In [ ]:
X_test.shape

(1, 3)

**OBS**.: O correto seria dividir em 3 subconjuntos, mas apenas para este exemplo, vamos usar dois subconjuntos.

## 4. Engenharia de atributos (exemplo simples)

Criar um atributo novo: salário por idade

In [ ]:
def salario_por_idade(X):
    X = X.copy()
    X["salario_idade"] = X["salario"] / X["idade"]
    return X

# Transforma a função acima em um "transformador" de dados que pode ser usado com outras classes da biblioteca SciKit-Learn.
feature_engineering = FunctionTransformer(salario_por_idade)

## 5. Pré-processamento (escalonamento + codificação)

In [ ]:
numeric_features = ["idade", "salario", "salario_idade"]
categorical_features = ["estado"]

# O ColumnTransformer aplica transformações por coluna especificada.
preprocessor = ColumnTransformer(
    transformers=[
        ("num", StandardScaler(), numeric_features),
        ("cat", OneHotEncoder(), categorical_features)
    ]
)

O `ColumnTransformer` permite aplicar transformações diferentes a colunas diferentes, tudo em um único objeto.

## 6. Seleção e extração de atributos

+ Seleção: remove variáveis com baixa variância

+ Extração: PCA (seleciona 2 componentes principais)

In [ ]:
feature_selection = VarianceThreshold(threshold=0.01)
feature_extraction = PCA(n_components=2)

## 7. Pipeline completo

**OBS.1**.: Usamos um modelo de classificação binária, o `Regressor logístico`.

**OBS.2**: Com a classe `Pipeline`, podemos combinar todas as etapas acima.


In [ ]:
pipeline = Pipeline(steps=[
    ("feature_engineering", feature_engineering),
    ("preprocessing", preprocessor),
    ("feature_selection", feature_selection),
    ("feature_extraction", feature_extraction),
    ("model", LogisticRegression())
])

## 8. Treinamento do modelo de ML

In [ ]:
pipeline.fit(X_train, y_train)

Pipeline(steps=[('feature_engineering',
                 FunctionTransformer(func=<function salario_por_idade at 0x7dd7f8c09b20>)),
                ('preprocessing',
                 ColumnTransformer(transformers=[('num', StandardScaler(),
                                                  ['idade', 'salario',
                                                   'salario_idade']),
                                                 ('cat', OneHotEncoder(),
                                                  ['estado'])])),
                ('feature_selection', VarianceThreshold(threshold=0.01)),
                ('feature_extraction', PCA(n_components=2)),
                ('model', LogisticRegression())])

## 9. Desempenho do modelo no conjunto de treinamento

In [ ]:
# Usamos o modelo treinado para realizar inferências.
y_train_pred = pipeline.predict(X_train)

# Cálculo da acurácia do conjunto de treinamento.
train_accuracy = accuracy_score(y_train, y_train_pred)

print(f"Acurácia no treinamento: {train_accuracy:.2f}")

Acurácia no treinamento: 0.50


## 10. Desempenho do modelo no conjunto de teste

In [ ]:
# Usamos o modelo treinado para realizar inferências.
y_test_pred = pipeline.predict(X_test)

# Cálculo da acurácia do conjunto de teste.
test_accuracy = accuracy_score(y_test, y_test_pred)

print(f"Acurácia no teste: {test_accuracy:.2f}")

Acurácia no teste: 0.00


## 10. Análise

Motivo principal do resultado ruim é devido ao **dataset ser extremamente pequeno**.

O modelo não consegue aprender o comportamento geral por trás dos dados

+ Existe apenas uma observação no conjunto de teste

+ Errar essa única previsão → acurácia = 0%

+ Acertar → acurácia = 100%


Com pouquíssimos dados:

+ O PCA não consegue estimar bem as direções de variância

+ Com isso, informação relevante pode ser descartada

+ O modelo é treinado com um conjunto mal transformado

+ O PCA exige mais dados para ser confiável.